# 03 — Evidence + Stack Ensemble

Validates the three bases' saved artifacts, projects each base's directional
prediction pairs into antisymmetric logit-space evidence (one observation per
physical match), trains a zero-intercept logistic stacker on the OOF evidence,
and logs the candidate run + manifest for 05.

Each 02 notebook saved its best model's directional OOF/test predictions
(`{class}_oof.parquet`, ``{class}_test.parquet``) plus tuning metric
(`{class}_score.json`); each artifact holds TWO rows per physical match.
A single deterministic orientation is chosen per match, then:

    evidence = (logit(clip(p_chosen)) - logit(clip(p_other))) / 2

so the stacker sees exactly one observation per physical match and no two
independent mirrors. No sklearn artifact path is assumed — the NN participates
through its saved predictions.


In [ ]:
from src.constants import (
    CANDIDATE_MANIFEST,
    DATA_PROCESSED,
    STACK_ORDER,
    load_env,
    RECENCY_HALF_LIFE_DAYS,
    RECENCY_HALF_LIFE_KEY,
    RECENCY_CUTOFF_KEY,
    MODELS_ARTIFACTS,
    MIN_TRAINING_DATE,
    TRAIN_FRACTION,
    VAL_FRACTION,
    TEST_FRACTION,
)

input_dir = str(DATA_PROCESSED)
output_dir = str(MODELS_ARTIFACTS)
random_state = 42
stacker_config = {"fit_intercept": False, "random_state": random_state}
model_names = list(STACK_ORDER)
candidate_manifest = str(CANDIDATE_MANIFEST)

load_env()

In [ ]:
import json
from typing import Any
import mlflow
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from src.evaluate.symmetry import antisymmetric_evidence, evidence_to_probability

In [ ]:
# ── Load labels and identity columns ──
info_train = pd.read_parquet(f"{input_dir}/info_train.parquet").reset_index(drop=True)
info_test = pd.read_parquet(f"{input_dir}/info_test.parquet")
train_cutoff = max(info_train["match_date"].max(), info_test["match_date"].max())
y_train = pd.read_parquet(f"{input_dir}/y_train.parquet")["y"].reset_index(drop=True)
y_test = pd.read_parquet(f"{input_dir}/y_test.parquet")["y"].reset_index(drop=True)
info_val = pd.read_parquet(f"{input_dir}/info_val.parquet").reset_index(drop=True)
y_val = pd.read_parquet(f"{input_dir}/y_val.parquet")["y"].reset_index(drop=True)
print(f"Train: {len(y_train)} rows ({info_train['match_id'].nunique()} matches)")
print(f"Test:  {len(y_test)} rows ({info_test['match_id'].nunique()} matches)")
print(f"Val:   {len(y_val)} rows ({info_val['match_id'].nunique()} matches)")

In [ ]:
# ── Directional base predictions -> antisymmetric evidence (one row per match) ──
# Each 02 artifact has TWO rows per physical match (both orientations). For
# every match we pick ONE deterministic evaluation orientation
ID_COLS = ["match_id", "player_id", "opponent_id"]


def _load_bases(suffix, require_fold):
    """Load the three bases' directional {suffix} predictions and validate that
    all three share identical identity (and fold) sequences."""
    frames = {}
    for name in model_names:
        df = pd.read_parquet(f"{output_dir}/{name}_{suffix}.parquet")
        expected = ID_COLS + (["fold"] if require_fold else []) + ["pred"]
        if list(df.columns) != expected:
            raise ValueError(
                f"{name}_{suffix}.parquet columns must be {expected}, got {list(df.columns)}"
            )
        counts = df["match_id"].value_counts()
        bad = int((counts != 2).sum())
        if bad:
            raise ValueError(
                f"{name}_{suffix}.parquet: {bad} match_id(s) do not appear exactly twice"
            )
        frames[name] = df
    ref = frames[model_names[0]]
    for name in model_names[1:]:
        for col in ID_COLS + (["fold"] if require_fold else []):
            if not (frames[name][col].astype(str) == ref[col].astype(str)).all():
                raise ValueError(
                    f"{name}_{suffix} {col} sequence differs from {model_names[0]}_{suffix}"
                )
    return frames


def _assert_mirrored(df: pd.DataFrame, label: str) -> None:
    """Each match_id's two rows must be swapped orientations (A,B)/(B,A)."""
    frame: Any = df
    pairs_by_match = {}
    for raw_row in pd.DataFrame.to_dict(frame, orient="records"):
        row: Any = raw_row
        pairs_by_match.setdefault(row.get("match_id"), []).append(
            (str(row.get("player_id")), str(row.get("opponent_id")))
        )

    for match_id, rows in pairs_by_match.items():
        first, second = rows
        first_player, first_opponent = first
        second_player, second_opponent = second
        if not ((first_player, first_opponent) == (second_opponent, second_player)):
            raise ValueError(
                f"{label}: match {match_id} rows are not mirrored orientations: {rows}"
            )


def _evidence_and_eval(frames, labels, suffix, match_meta=None):
    """One antisymmetric evidence row per physical match in the chosen
    orientation (lexicographically smaller player_id), plus that row's label.
    When match_meta is provided (expanded evidence), attach match_date and
    fold from the match-level metadata."""
    ref = frames[model_names[0]].reset_index(drop=True)
    chosen_mask = (ref["player_id"].astype(str) <= ref["opponent_id"].astype(str)).to_numpy()
    n_matches = ref["match_id"].nunique()
    if chosen_mask.sum() != n_matches:
        raise ValueError(
            f"{suffix}: expected exactly one chosen row per match, "
            f"got {chosen_mask.sum()} for {n_matches} matches"
        )
    match_ids = ref.loc[chosen_mask, "match_id"].to_numpy()
    y_chosen = labels.iloc[np.flatnonzero(chosen_mask)].to_numpy()
    order = np.argsort(match_ids, kind="stable")
    evidence = {}
    for name in model_names:
        df = frames[name].reset_index(drop=True)
        if not (
            df.loc[chosen_mask, "match_id"].to_numpy()
            == df.loc[~chosen_mask, "match_id"].to_numpy()
        ).all():
            raise ValueError(f"{name}_{suffix}: chosen/other rows are not paired per match")
        p_chosen = df.loc[chosen_mask, "pred"].to_numpy()
        p_other = df.loc[~chosen_mask, "pred"].to_numpy()
        evidence[name] = np.asarray(antisymmetric_evidence(p_chosen, p_other))[order]
    eval_df = pd.DataFrame({"match_id": match_ids[order], "match_won": y_chosen[order]})
    if match_meta is not None:
        meta = match_meta.set_index("match_id")
        ev_ids = eval_df["match_id"].astype(str)
        eval_df["match_date"] = meta.loc[ev_ids, "match_date"].to_numpy()
        eval_df["fold"] = meta.loc[ev_ids, "fold"].to_numpy()
        eval_df = eval_df[["match_id", "match_date", "fold", "match_won"]]
    return pd.DataFrame(evidence), eval_df


# ── Train-only OOF + final-refit TEST evidence ──
# These keep the existing base-AUC diagnostics and the final-refit test path
# intact; only the expanded train+validation evidence below feeds the stacker.
for kind, info, labels, require_fold in (
    ("oof", info_train, y_train, True),
    ("test", info_test, y_test, False),
):
    frames = _load_bases(kind, require_fold)

    info_ref = info[ID_COLS].astype(str).to_numpy()
    for name in model_names:
        if not (frames[name][ID_COLS].astype(str).to_numpy() == info_ref).all():
            raise ValueError(f"{name}_{kind} identity sequence differs from info_{kind} order")
    evidence, eval_df = _evidence_and_eval(frames, labels, kind)
    if kind == "oof":
        oof_evidence, oof_eval = evidence, eval_df
        print(f"oof:  {len(eval_df)} matches -> evidence held in memory ({list(evidence.columns)})")
    else:
        test_evidence, test_eval = evidence, eval_df
        evidence.to_parquet(f"{output_dir}/test_evidence.parquet", index=False)
        eval_df.to_parquet(f"{output_dir}/test_eval.parquet", index=False)
        print(
            f"test: {len(eval_df)} matches -> test_evidence.parquet ({list(evidence.columns)}) + test_eval.parquet"
        )

# ── Expanded train+validation evidence ──
TV_SUFFIX = "train_val_oof"
tv_frames = _load_bases(TV_SUFFIX, require_fold=True)

# Labels and dates keyed by (match_id, player_id, opponent_id) so the base
# artifact row order does not have to match the train/val concatenation order.
tv_info = pd.concat([info_train, info_val], ignore_index=True)
tv_y = pd.concat([y_train, y_val], ignore_index=True)
tv_label_lookup = (
    tv_info[["match_id", "player_id", "opponent_id"]]
    .copy()
    .assign(y=tv_y.to_numpy())
    .set_index(["match_id", "player_id", "opponent_id"])["y"]
)

# The expanded base artifacts' fold assignment is read from the train+val
# metadata to attach match_date/fold during evidence assembly below.
tv_assignment = pd.read_parquet(f"{input_dir}/train_val_fold_assignment.parquet")
ref = tv_frames[model_names[0]].reset_index(drop=True)
ref_lookup_df = tv_label_lookup.reset_index()
ref_y = ref[["match_id", "player_id", "opponent_id"]].merge(
    ref_lookup_df, on=["match_id", "player_id", "opponent_id"], how="left"
)
tv_labels = ref_y["y"].reset_index(drop=True).astype(int)
tv_evidence, tv_eval = _evidence_and_eval(tv_frames, tv_labels, TV_SUFFIX, match_meta=tv_assignment)

# Assemble the exact persisted schema from the evidence and per-match eval.
train_val_evidence = tv_eval.copy()
for name in model_names:
    train_val_evidence[name] = tv_evidence[name].to_numpy()
train_val_evidence = train_val_evidence[
    ["match_id", "match_date", "fold", "match_won", *model_names]
]

train_val_evidence.to_parquet(f"{output_dir}/train_val_evidence.parquet", index=False)
print(
    f"train_val: {len(train_val_evidence)} matches -> train_val_evidence.parquet "
    f"({list(train_val_evidence.columns)})"
)

for name in model_names:
    with open(f"{output_dir}/{name}_score.json") as f:
        score = json.load(f)
    print(f"Best {name:6s}: {score['metric']} = {score['score']:.4f}")

In [ ]:
# ── Consolidate pinned base-model identities from 02 ──
# Each 02 notebook wrote {name}_model_version.json with the name to register
# under at promotion, the run ID, and the run-artifact model URI.
base_pins = {}
for name in model_names:
    with open(f"{output_dir}/{name}_model_version.json") as f:
        base_pins[name] = json.load(f)
# Tabular-only lineage: no auxiliary (bio) artifacts are pinned, so aux_pins is empty.
aux_pins: dict[str, str] = {}
for name, pin in base_pins.items():
    print(f"  {name}: {pin['registered_model_name']} (run {pin['run_id'][:8]})")

In [ ]:
# ── Compare best-per-class ROC-AUC (OOF, chosen-orientation evidence) ──
# Each base's symmetric probability sigmoid(evidence) vs the chosen
# orientation label — one observation per physical match.
for name in model_names:
    score = roc_auc_score(oof_eval["match_won"], evidence_to_probability(oof_evidence[name]))
    print(f"  {name:8s} OOF ROC-AUC: {score:.4f}")

In [ ]:
# ── Train the no-intercept stacker on the train+validation evidence ──


def _require_evidence_columns(df, label):
    cols = list(df.columns)
    if len(cols) != len(set(cols)):
        raise ValueError(f"{label}: duplicate evidence columns: {cols}")
    if set(cols) != set(model_names):
        raise ValueError(f"{label}: evidence columns must be exactly {model_names}, got {cols}")
    return df[model_names].reset_index(drop=True)


expanded_evidence = _require_evidence_columns(train_val_evidence[model_names], "train_val_evidence")
if list(train_val_evidence.columns)[:4] != ["match_id", "match_date", "fold", "match_won"]:
    raise ValueError(
        f"train_val_evidence must start with [match_id, match_date, fold, match_won], "
        f"got {list(train_val_evidence.columns)}"
    )
if not train_val_evidence["match_id"].is_unique:
    raise ValueError("train_val_evidence must hold exactly one row per physical match")
if len(train_val_evidence) != len(expanded_evidence):
    raise ValueError(
        f"train_val_evidence rows ({len(train_val_evidence)}) differ from evidence rows ({len(expanded_evidence)})"
    )
y_expanded = train_val_evidence["match_won"].to_numpy()

meta = LogisticRegression(**stacker_config)
meta.fit(expanded_evidence, y_expanded)

# Assert the no-intercept contract: the fitted model has a zero intercept.
assert not meta.fit_intercept
assert np.allclose(meta.intercept_, 0.0)

print(
    f"Expanded evidence shape: {expanded_evidence.shape} (columns {list(expanded_evidence.columns)})"
)
print("Meta-model coefficients (evidence -> logit):")
coef_values = np.asarray(meta.coef_).reshape(-1).tolist()
for name, coef in zip(model_names, coef_values, strict=False):
    print(f"  {name:8s} {coef:.4f}")

In [ ]:
# ── Strictly forward (causal) stacker calibration evidence ──
# For each fold AFTER the first base-predicted fold, fit a temporary
# zero-intercept logistic stacker on ONLY earlier evidence folds

tv_ev = pd.read_parquet(f"{output_dir}/train_val_evidence.parquet")
tv_ev = tv_ev.assign(fold=tv_ev["fold"].astype(int))

ev_cols = _require_evidence_columns(tv_ev[model_names], "train_val_evidence")
y_tv = tv_ev["match_won"].to_numpy()

fold_arr = tv_ev["fold"].to_numpy()
folds = sorted(np.unique(fold_arr).tolist())
first_fold = folds[0]
parts = []
for f in folds:
    pred_mask = fold_arr == f
    if f == first_fold:
        # First base-predicted fold: no earlier stacker training evidence.
        print(f"  omit fold {f}: first base-predicted fold (no earlier fit evidence)")
        continue
    fit_mask = fold_arr < f
    fit_folds = np.unique(fold_arr[fit_mask])
    stack_f = LogisticRegression(**stacker_config)
    stack_f.fit(ev_cols.iloc[fit_mask], y_tv[fit_mask])
    preds = stack_f.predict_proba(ev_cols.iloc[pred_mask])[:, 1]
    part = tv_ev.iloc[pred_mask][["match_id", "match_date", "fold"]].copy()
    part["stack_pred_cv"] = preds
    part["match_won"] = y_tv[pred_mask]
    parts.append(part)
    print(
        f"  fold {f}: fit on earlier folds {fit_folds.tolist()} -> {int(pred_mask.sum())} predictions"
    )

train_val_stack_cv = pd.concat(parts, ignore_index=True)
train_val_stack_cv = train_val_stack_cv[
    ["match_id", "match_date", "fold", "stack_pred_cv", "match_won"]
]
train_val_stack_cv.to_parquet(f"{output_dir}/train_val_stack_cv.parquet", index=False)
print(
    f"train_val_stack_cv.parquet: {train_val_stack_cv.shape[0]} matches, "
    f"folds {sorted(train_val_stack_cv['fold'].unique().tolist())}"
)

In [ ]:
# ---- Cross-fitted stacker predictions for calibration ----
# The final `meta` stacker above is fit on ALL OOF evidence
oof_ref = pd.read_parquet(f"{output_dir}/{model_names[0]}_oof.parquet")
chosen_mask = (oof_ref["player_id"].astype(str) <= oof_ref["opponent_id"].astype(str)).to_numpy()
if chosen_mask.sum() != len(oof_eval):
    raise ValueError(
        f"expected one chosen OOF row per match, got {chosen_mask.sum()} for {len(oof_eval)}"
    )
fold_per_match = oof_ref.groupby(oof_ref["match_id"].astype(str))["fold"].nunique()
if not (fold_per_match == 1).all():
    raise ValueError("each match must carry exactly one fold across both orientations")
fold_map = dict(
    zip(
        oof_ref.loc[chosen_mask, "match_id"].astype(str),
        oof_ref.loc[chosen_mask, "fold"],
        strict=False,
    )
)
oof_fold = oof_eval["match_id"].astype(str).map(fold_map)
if oof_fold.isna().any():
    raise ValueError("every oof_eval match_id must resolve to exactly one fold")
oof_fold = oof_fold.to_numpy(dtype=int)
assert oof_fold.shape == (len(oof_evidence),)

stack_cv_preds = np.empty(len(oof_evidence), dtype=float)
oof_labels = oof_eval["match_won"].to_numpy()
for f in np.unique(oof_fold):
    train_mask = oof_fold != f
    val_mask = oof_fold == f
    stack_f = LogisticRegression(**stacker_config)
    stack_f.fit(oof_evidence.iloc[train_mask], oof_labels[train_mask])
    stack_cv_preds[val_mask] = stack_f.predict_proba(oof_evidence.iloc[val_mask])[:, 1]

assert set(np.unique(oof_fold)) == set(fold_map.values()), "folds must span the OOF fold set"
assert np.isfinite(stack_cv_preds).all(), "stack_cv_preds must be finite"
assert (stack_cv_preds >= 0.0).all() and (stack_cv_preds <= 1.0).all(), (
    "stack_cv_preds must lie in [0,1]"
)

stack_cv = pd.DataFrame(
    {
        "match_id": oof_eval["match_id"].to_numpy(),
        "stack_pred_cv": stack_cv_preds,
        "fold": oof_fold,
    }
)
stack_cv.to_parquet(f"{output_dir}/oof_stack_cv.parquet", index=False)
print(f"oof_stack_cv.parquet: {stack_cv.shape[0]} matches")
print(stack_cv["fold"].value_counts().sort_index().to_string())

In [ ]:
# ── Log pinned candidate to MLflow (no registration/promotion) ──
mlflow.set_experiment("ensemble-training")
mlflow.set_experiment_tag("pipeline", "tune")
with mlflow.start_run(run_name="fit-ensemble-lr", tags={"pipeline": "tune"}):
    for name in model_names:
        pin = base_pins[name]
        mlflow.log_param(f"base_{name}_registered_name", pin["registered_model_name"])
        mlflow.log_param(f"base_{name}_run_id", pin["run_id"])
        mlflow.log_param(f"base_{name}_model_uri", pin["model_uri"])
    mlflow.log_metrics(
        {f"weight_{name}": coef for name, coef in zip(model_names, coef_values, strict=False)}
    )
    # The stacker is fitted with fit_intercept=False; log the zero intercept
    # explicitly so the metadata records the antisymmetry contract.
    mlflow.log_metric("intercept", 0.0)
    mlflow.log_param(RECENCY_HALF_LIFE_KEY, RECENCY_HALF_LIFE_DAYS)
    mlflow.log_param(RECENCY_CUTOFF_KEY, str(pd.Timestamp(train_cutoff).date()))
    stacked_info = mlflow.sklearn.log_model(meta, name="stacked_ensemble")
    run = mlflow.active_run()
    assert run is not None
    run_id = run.info.run_id
    print(f"Candidate logged to MLflow run: {run_id}")

In [ ]:
# ── Candidate handoff manifest: exact candidate run + full lineage ──
manifest = {
    "candidate_run_id": run_id,
    "model_uri": stacked_info.model_uri,
    "artifact_path": "stacked_ensemble",
    "base_pins": base_pins,
    "aux_pins": aux_pins,
    "min_training_date": MIN_TRAINING_DATE.isoformat(),
    "train_fraction": TRAIN_FRACTION,
    "val_fraction": VAL_FRACTION,
    "test_fraction": TEST_FRACTION,
    "recency_half_life_days": RECENCY_HALF_LIFE_DAYS,
    "recency_cutoff_date": str(pd.Timestamp(train_cutoff).date()),
}
with open(candidate_manifest, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest written to {candidate_manifest}")
print(f"  candidate_run_id: {run_id}")
print(f"  model_uri:        {stacked_info.model_uri}")
print(f"  lineage: {len(base_pins)} base classes, {len(aux_pins)} aux keys")